# PCB Component Detection & SAM Point Extractor

Pipeline workflow:
1. **Download Kaggle Datasets**: Fetch dataset via `kagglehub`.
2. **Label Homogenizer**: Standardize labels into KiCad footprint classes.
3. **SAM Point Extractor**: Generate `(x, y)` polygon points from SAM masks.
4. **Multi-Format Exporter**: Export to YOLO-seg `.txt`, LabelMe `.json`, and `polygon_points.csv`.

In [ ]:
# Step 1: Install Dependencies
!pip install torch torchvision opencv-python matplotlib pandas kagglehub git+https://github.com/facebookresearch/segment-anything.git

In [ ]:
import os
import cv2
import torch
import random
import json
import pandas as pd
import numpy as np
import urllib.request
import matplotlib.pyplot as plt
from pathlib import Path
from segment_anything import sam_model_registry, SamPredictor
import kagglehub

# KiCad Standard Classes
KICAD_CLASSES = [
    'Capacitor_SMD',
    'Resistor_SMD',
    'Package_SO',
    'Package_TO_SOT_SMD',
    'Diode_SMD',
    'Connector',
    'Inductor_SMD',
    'Button_Switch_SMD',
    'LED_SMD',
    'Transformer_SMD',
    'PCB_Defect',
    'Unknown_Component'
]

LABEL_SYNONYMS = {
    'Capacitor_SMD': ['c', 'cap', 'capacitor', 'capacitors', 'c_smd', 'c_tht', 'cap1', 'cap2', 'cap3', 'cap4'],
    'Resistor_SMD': ['r', 'res', 'resistor', 'resistors', 'r_smd', 'r_tht'],
    'Package_SO': ['ic', 'chip', 'integrated_circuit', 'soic', 'sop', 'qfp', 'qfn', 'dip', 'mcu'],
    'Package_TO_SOT_SMD': ['q', 'transistor', 'mosfet', 'fet', 'bjt', 'sot', 'sot23', 'to220', 'dopak', 'mov'],
    'Diode_SMD': ['d', 'diode', 'diodes', 'zener', 'schottky', 'tvs'],
    'Connector': ['conn', 'connector', 'connectors', 'header', 'plug', 'jack', 'usb', 'terminal'],
    'Inductor_SMD': ['l', 'ind', 'inductor', 'choke', 'coil'],
    'Button_Switch_SMD': ['sw', 'switch', 'button', 'btn', 'tactile', 'toggle'],
    'LED_SMD': ['led', 'leds', 'light_emitting_diode'],
    'Transformer_SMD': ['t', 'transformer', 'xfmr'],
    'PCB_Defect': ['defect', 'open', 'short', 'mousebite', 'spur', 'pin-hole', 'spurious_copper']
}

class LabelHomogenizer:
    def __init__(self):
        self.lookup = {}
        for std_name, synonyms in LABEL_SYNONYMS.items():
            for syn in synonyms:
                self.lookup[syn.lower()] = std_name
                
    def homogenize(self, raw_label):
        if isinstance(raw_label, int):
            return KICAD_CLASSES[raw_label] if raw_label < len(KICAD_CLASSES) else f"Class_{raw_label}"
        cleaned = str(raw_label).strip().lower()
        if cleaned in self.lookup:
            return self.lookup[cleaned]
        for syn, std_name in self.lookup.items():
            if syn in cleaned:
                return std_name
        return f"Unknown_{raw_label}"

homogenizer = LabelHomogenizer()

# Initialize SAM Predictor
sam_checkpoint = Path(r"C:\Users\ANAGHA\sam_vit_b.pth")
sam_url = "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth"
if not sam_checkpoint.exists():
    sam_checkpoint.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(sam_url, str(sam_checkpoint))

device = 'cuda' if torch.cuda.is_available() else 'cpu'
sam = sam_model_registry["vit_b"](checkpoint=str(sam_checkpoint))
sam.to(device=device)
sam.eval()
predictor = SamPredictor(sam)

In [ ]:
# Step 2: Download Kaggle Dataset
dataset_path = kagglehub.dataset_download("ficslab/fics-pcb")
print("Downloaded Kaggle dataset to:", dataset_path)

In [ ]:
# Step 3: Extract SAM Points & Export to LabelMe JSON
def export_to_labelme_json(image_name, img_w, img_h, polygon_records, output_json_path):
    shapes = []
    for rec in polygon_records:
        raw_lbl = rec.get('raw_label', rec.get('class_id'))
        kicad_name = homogenizer.homogenize(raw_lbl)
        pts_norm = rec['points']
        pixel_pts = [[round(pts_norm[i] * img_w, 2), round(pts_norm[i+1] * img_h, 2)] for i in range(0, len(pts_norm), 2)]
        shapes.append({
            "label": kicad_name,
            "points": pixel_pts,
            "group_id": None,
            "shape_type": "polygon",
            "flags": {}
        })
    labelme_data = {
        "version": "5.0.1",
        "flags": {},
        "shapes": shapes,
        "imagePath": image_name,
        "imageHeight": img_h,
        "imageWidth": img_w
    }
    with open(output_json_path, "w", encoding="utf-8") as f:
        json.dump(labelme_data, f, indent=2)
    return output_json_path